# sysone quickstart: accuracy before and after training

This notebook trains a small decision model on synthetic support tickets and measures accuracy
on held-out tickets **before** and **after** training, for each question type:

- **choice**: which team should handle the ticket (3 options)
- **score**: how severe it is (4 levels)
- **noul**: is the customer asking for a refund (yes / no)

It uses `tuning="head"`: the base model stays frozen and only the small decision head trains.
Expect a few minutes on a Mac; the first run also downloads the base model.

In [ ]:
import random

import pandas as pd

from sysone import DecoderModel, EncoderModel

# "qwen" -> Qwen/Qwen3-0.6B (about 1.2 GB download), "modernbert" -> answerdotai/ModernBERT-base (about 600 MB)
MODEL = "qwen"
Model = DecoderModel if MODEL == "qwen" else EncoderModel

## Data

Each ticket mentions a topic (which decides the team), a severity phrase, and in about half of
them a refund request.
Rows are plain dicts: a `state` (the text), the `questions` to answer about it, and the
`expected` answers. `category` is optional; here it is the team.

In [ ]:
TEAMS = {
    "sales": ["a quote", "pricing", "a discount", "a contract renewal"],
    "operations": ["a delayed shipment", "the warehouse", "a tracking number", "a late delivery"],
    "servicing": ["an exchange", "a broken unit", "a repair", "returning the item"],
}
SEVERITY = ["none", "minor", "major", "critical"]
SEVERITY_WORDS = ["just a question", "a small issue", "a serious problem", "an emergency, everything is down"]
OPENERS = ["Hi, this is about", "Hello team, writing about", "Quick note regarding", "Urgent message about"]
REFUND_ASKS = ["I want my money back.", "Please refund the order.", "Can I get a refund?"]

QUESTIONS = {
    "team": {"type": "choice", "instructions": "Which team should handle it?", "criteria": list(TEAMS)},
    "severity": {"type": "score", "instructions": "How severe is it?", "criteria": SEVERITY},
    "refund": {"type": "noul", "instructions": "Is the customer asking for a refund?"},
}


def make_ticket(rng, i):
    team = rng.choice(list(TEAMS))
    topic = rng.choice(TEAMS[team])
    level = rng.randrange(len(SEVERITY))
    refund = rng.random() < 0.5
    text = "Ticket %d: %s %s. It is %s." % (i, rng.choice(OPENERS), topic, SEVERITY_WORDS[level])
    if refund:
        text += " " + rng.choice(REFUND_ASKS)
    return {
        "state": text,
        "category": team,
        "questions": QUESTIONS,
        "expected": {"team": team, "severity": level, "refund": refund},
    }


rng = random.Random(0)
train = [make_ticket(rng, i) for i in range(400)]
test = [make_ticket(rng, 400 + i) for i in range(150)]  # held out: never seen in training
print(len(train), "training tickets,", len(test), "test tickets")
train[0]

## Scoring

`predict` returns answers in the Jev shape. A choice is right if `choice` matches; a score is right
if the expected level, rounded, matches; a noul is right if `noul` (P(yes)) is on the correct side
of 0.5.

In [ ]:
def accuracy(model, rows):
    preds = model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    hits = {"team (choice)": 0, "severity (score)": 0, "refund (noul)": 0}
    for row, pred in zip(rows, preds):
        a, e = pred["answers"], row["expected"]
        hits["team (choice)"] += a["team"]["choice"] == e["team"]
        hits["severity (score)"] += round(a["severity"]["score"]) == e["severity"]
        hits["refund (noul)"] += (a["refund"]["noul"] > 0.5) == e["refund"]
    return {name: n / len(rows) for name, n in hits.items()}


# Chance level: always guessing the most common answer in the test set
def majority_baseline(rows):
    out = {}
    for name, key in (("team (choice)", "team"), ("severity (score)", "severity"), ("refund (noul)", "refund")):
        values = [r["expected"][key] for r in rows]
        out[name] = max(values.count(v) for v in set(values)) / len(values)
    return out


results = {"chance (majority answer)": majority_baseline(test)}
pd.DataFrame(results).T.round(3)

## Before training

A model must be trained before `predict` will run: an untrained decision head gives noise, and
sysone refuses rather than return it. To see that noise anyway, we run one epoch with a learning
rate of 0, which leaves the randomly initialised head unchanged.

In [ ]:
untrained = Model(tuning="head", epochs=1, head_lr=0.0, verbose=0).fit(train)
results["before training (random head)"] = accuracy(untrained, test)
pd.DataFrame(results).T.round(3)

## Train

The backbone runs once over the training tickets and its outputs are cached; each epoch then
trains only the head. 10% of the tickets are held back for validation and early stopping.

In [ ]:
model = Model(tuning="head", batch_size=32)
model.fit(train)
model.report_["validation"]

## After training

In [ ]:
results["after training"] = accuracy(model, test)
pd.DataFrame(results).T.round(3)

## Try it yourself

Questions are passed at prediction time, so you can ask any of them about any text.
`probabilities` shows how sure the model is.

In [ ]:
model.predict("Ticket 9001: Urgent message about a broken unit. It is an emergency, everything is down. "
              "Can I get a refund?", QUESTIONS)

## Save and load

A saved model stores only the trained head and its configuration (a few hundred KB); the base
model is reloaded from the Hugging Face cache.

In [ ]:
model.save("quickstart-model")
reloaded = Model.load("quickstart-model", verbose=0)
assert accuracy(reloaded, test) == results["after training"]
print("reloaded model gives the same test accuracy")